# Exercice partie 1 - Comprenez les bénéfices du NoSQL

## Etape 1 : Importez les données 

Voir les diapositives de mon template de présentation pour les différentes étapes et copies d'écran de l'importation de données

### Etape 2 : Comprenez les données 

In [41]:
from pymongo import MongoClient

In [42]:
client = MongoClient("mongodb://localhost:27017")
db = client["noscites"]
collection = db["listings_paris"]

Etant donné que j'avais un doute sur le fait que l'attribut "license" concerne un hote ou un logement, j'ai essayé de vérifier si un hote pouvait avoir plusieurs licenses en fonction de ses logements. Ce qui est bien le cas à priori avec les résultats ci-dessous. Donc on a attribué l'attribut license à la catégorie 'logement' et non 'hote'

In [43]:
# Vérifier s'il existe des hôtes avec des licences DIFFÉRENTES sur leurs annonces
pipeline = [
    { "$match": { "license": { "$ne": "" } } },
    { "$group": {
        "_id": "$host_id",
        "host_name": { "$first": "$host_name" },
        "nb_licences_distinctes": { "$addToSet": "$license" }
    }},
    { "$match": { "$expr": { "$gt": [ { "$size": "$nb_licences_distinctes" }, 1 ] } } },
    { "$limit": 3 }
]
result = list(collection.aggregate(pipeline))
print(f"Hôtes avec licences différentes sur leurs annonces : {len(result)}")
for r in result:
    print(f"{r['host_name']} : {r['nb_licences_distinctes']}")

Hôtes avec licences différentes sur leurs annonces : 3
Silène : [7511105565802, 7511105970529]
Romain : [7511810590902, 7511807983283]
Roger : [7511313669404, 7511913668545, 7511713669094]


### Requête 1 : Nombre total de documents

In [44]:
total = collection.count_documents({})
print(f"Nombre total de documents : {total}")

Nombre total de documents : 95885


### Requête 2 : Logements avec disponibilités

In [46]:
disponibles = collection.count_documents({"has_availability" : "t"})
print(f"Logements avec disponibilités : {disponibles}")

Logements avec disponibilités : 90173


In [47]:
# Sur un seul document
doc = collection.find_one()
print(f"Nombre de champs : {len(doc)}")

# Distribution sur toute la collection
pipeline = [
    { "$project": { "nbChamps": { "$size": { "$objectToArray": "$$ROOT" } } } },
    { "$group": { "_id": "$nbChamps", "count": { "$sum": 1 } } },
    { "$sort": { "_id": 1 } }
]
for r in collection.aggregate(pipeline):
    print(f"{r['count']} documents ont {r['_id']} champs")

Nombre de champs : 76
95885 documents ont 76 champs


# Exercice partie 2 - Analysez une base de données NoSQL

## Etape 1 : Requêtez les données avec le CLI (Command Line Interface)

### 1. Combien d'annonces par type de location ?

In [11]:
pipeline = [
    { "$group": { "_id": "$room_type", "count": { "$sum": 1 } } },
    { "$sort": { "count": -1 } }
]
for r in collection.aggregate(pipeline):
    print(f"{r['_id']} : {r['count']} annonces")

Entire home/apt : 85733 annonces
Private room : 8975 annonces
Hotel room : 776 annonces
Shared room : 401 annonces


Donc la logique de cette requête, si je ne me trompe pas, est la suivante :
- On groupe les individus sur l'attribut room_type.
- On affiche comme résultat de cet aggrégation la somme de chaque aggregat et on appelle de champs 'sum'
- On trie les résultats de manière décroissante (-1) grâce à sort.

### 2. Les 5 annonces avec le plus d'évaluations ?

In [12]:
resultats = collection.find(
    {},
    { "name": 1, "number_of_reviews": 1, "_id": 0 }
).sort("number_of_reviews", -1).limit(5)

for r in resultats:
    print(f"{r['name']} : {r['number_of_reviews']} évaluations")

Sweet & cosy room next to Canal Saint Martin ❤️ : 3067 évaluations
Double/Twin Room, close to Opera and the Louvre with breakfast included : 2620 évaluations
Bed in Dorm of 8 Beds "The Big One" in Paris : 2294 évaluations
Comfortable bed in shared rooms of 8 in Paris 12e : 2105 évaluations
Nice Room for 2 people : 2048 évaluations


La logique de cette requête est la suivante :
- On ne filtre pas les individus - on les prend tous ({}).
- On projette seulement les champs 'name' et 'number_of_reviews'
- On trie sur le champ 'number_of_review' nouvellement crée.
- On stocke le résultat (collection) dans une variable
- On affiche les élements un par un

### 3. Nombre total d'hôtes différents ?

In [15]:
total_hotes = len(collection.distinct("host_id"))
print(f"Nombre d'hôtes différents : {total_hotes}")

Nombre d'hôtes différents : 71979


La logique de cette requête est la suivante :
- On selectionne les noms d'hotes différents (par la fonction distinct).
- On calculer la taille de la collection sauvegardée.

### 4. Nombre de locations réservables instantanément + proportion ?

In [16]:
total = collection.count_documents({})
instant = collection.count_documents({ "instant_bookable": "t" })
proportion = (instant / total) * 100
print(f"Réservables instantanément : {instant}")
print(f"Proportion : {proportion:.2f}%")

Réservables instantanément : 22094
Proportion : 23.04%


La logique de cette requête est la suivante :
- On compte le nombre de logements disponibles tout de suite.
- On compte le nombre de logements total.
- On divise l'un par l'autre pour trouver la proportion.

### 5. Hôtes avec plus de 100 annonces + pourcentage ?

In [17]:
pipeline = [
    { "$group": { "_id": "$host_id", "host_name": { "$first": "$host_name" }, "nb_annonces": { "$sum": 1 } } },
    { "$match": { "nb_annonces": { "$gt": 100 } } },
    { "$sort": { "nb_annonces": -1 } }
]
hotes = list(collection.aggregate(pipeline))

total_hotes = len(collection.distinct("host_id"))
proportion = (len(hotes) / total_hotes) * 100

for h in hotes:
    print(f"{h['host_name']} (ID {h['_id']}) : {h['nb_annonces']} annonces")
print(f"\n→ {len(hotes)} hôtes concernés ({proportion:.2f}% des hôtes)")

Blueground (ID 314994947) : 730 annonces
Veeve (ID 33889201) : 497 annonces
Pierre De WeHost (ID 50502817) : 426 annonces
Sébastien (ID 50978178) : 307 annonces
Cédric De ClickYourFlat (ID 26981054) : 274 annonces
FlexLiving (ID 460047164) : 232 annonces
Ludovic (ID 7642792) : 211 annonces
David Et Warren (ID 436103373) : 210 annonces
Checkmyguest (ID 528015349) : 189 annonces
Welkeys (ID 125797498) : 170 annonces
Studioprestige (ID 335998296) : 154 annonces
Sweet Inn (ID 51567288) : 149 annonces
IntoParis (ID 1112584) : 145 annonces
Rusard (ID 564251645) : 137 annonces
Parisian Home (ID 440295601) : 132 annonces
Barnes (ID 517515174) : 122 annonces
Jérémy (ID 99040006) : 120 annonces
Michael & Johanna (ID 28313443) : 119 annonces
Giacomo (ID 24495283) : 111 annonces
Check My Guest (ID 506389460) : 109 annonces
Checkmyguest (ID 374552379) : 104 annonces
Checkmyguest (ID 512746089) : 104 annonces
Check My Guest (ID 499962530) : 103 annonces
Pierre (ID 21630783) : 102 annonces

→ 24 hôte

La logique de cette requête est la suivante :
- On groupe par l'id de l'hote et son nom (car on veut le nom dans le résultat mais un nom seulement peut-être emprunté par plusieurs hotes différents)
- On calcule la somme comme fonction d'agregation qu'on nombre nb_annonces.
- On filtre en ne gardant seulement que les hotes dont le nombre d'annonces (nb_annonces) est plus grand ou égal à 100.
- On trie de manière décroissante sur le nombre d'annonce.
- On stocke le résultat dans une liste fait à partir d'une collection et on affiche les hotes un par un. 

### 6. Nombre de super hôtes + pourcentage ?

In [18]:
total_hotes = len(collection.distinct("host_id"))
super_hotes = len(collection.distinct("host_id", { "host_is_superhost": "t" }))
proportion = (super_hotes / total_hotes) * 100
print(f"Super hôtes : {super_hotes}")
print(f"Proportion : {proportion:.2f}% des hôtes")

Super hôtes : 10027
Proportion : 13.93% des hôtes


La logique de cette requête est la suivante :
- On récupère tous les hotes
- On récupère les hotes qui sont marqués comme 'super hotes' (par le booléen host_is_superhost : true).
- On divise l'un par l'autre pour obtenir la proportion.

## Etape 2 : Utilisez Polars pour des requêtes complexes

In [21]:
import polars as pl

In [23]:
df = pl.DataFrame(
    list(collection.find({}, {"_id": 0})),
    infer_schema_length=None  # analyse TOUTES les lignes pour inférer le schéma (et éviter les problèmes)
)

In [25]:
# Schéma complet (nom de colonne + type)
# Une colonne par ligne
for col, dtype in df.schema.items():
    print(f"{col} : {dtype}")

id : Int64
listing_url : String
scrape_id : Int64
last_scraped : String
source : String
name : String
description : String
neighborhood_overview : String
picture_url : String
host_id : Int64
host_url : String
host_name : String
host_since : String
host_location : String
host_about : String
host_response_time : String
host_response_rate : String
host_acceptance_rate : String
host_is_superhost : String
host_thumbnail_url : String
host_picture_url : String
host_neighbourhood : String
host_listings_count : String
host_total_listings_count : String
host_verifications : String
host_has_profile_pic : String
host_identity_verified : String
neighbourhood : String
neighbourhood_cleansed : String
neighbourhood_group_cleansed : String
latitude : Float64
longitude : Float64
property_type : String
room_type : String
accommodates : Int64
bathrooms : String
bathrooms_text : String
bedrooms : String
beds : String
amenities : String
price : String
minimum_nights : Int64
maximum_nights : Int64
minimum_mi

### 1. Taux de réservation moyen par mois par type de logement

In [27]:
taux_reservation = (
    df
    .with_columns([
        ((365 - pl.col("availability_365")) / 365 * 100)
        .alias("taux_reservation_pct"),
        ((365 - pl.col("availability_365")) / 12)
        .alias("jours_reserves_par_mois")
    ])
    .group_by("room_type")
    .agg([
        pl.col("taux_reservation_pct").mean().round(2).alias("taux_moyen_pct"),
        pl.col("jours_reserves_par_mois").mean().round(2).alias("jours_reserves_mois_moyen")
    ])
    .sort("taux_moyen_pct", descending=True)
)
print(taux_reservation)

shape: (4, 3)
┌─────────────────┬────────────────┬───────────────────────────┐
│ room_type       ┆ taux_moyen_pct ┆ jours_reserves_mois_moyen │
│ ---             ┆ ---            ┆ ---                       │
│ str             ┆ f64            ┆ f64                       │
╞═════════════════╪════════════════╪═══════════════════════════╡
│ Private room    ┆ 68.8           ┆ 20.93                     │
│ Entire home/apt ┆ 64.9           ┆ 19.74                     │
│ Shared room     ┆ 63.33          ┆ 19.26                     │
│ Hotel room      ┆ 52.38          ┆ 15.93                     │
└─────────────────┴────────────────┴───────────────────────────┘


La logique de cette requête est la suivante :
- On se sert des colonnes déjà présentes pour en créer d'autres (with column)
- Les deux colonnes crées sont taux_reservation_pc et jours_reserves_par_mois grâce aux calculs effectués sur les autres colonnes
- On groupe par le type de locations (group by)
- Sur lequel on utilise des fonctions d'aggregations (.agg) seulement sur les deux colonnes nouvellements crées (donc ce sont les seul qui restent à la fin) pour obtenir leur moyennes respectives.
- Enfin on trie les résultats par le taux_moyen_pct. Ce qui nous laisse 3 colonnes avec le 'room_type' qui a permi le group by.

### 2. Médiane du nombre d'avis pour tous les logements

In [30]:
# Avec Polars
mediane_avis = df.select(
    pl.col("number_of_reviews").median()
).item()
print(f"Médiane du nombre d'avis : {mediane_avis}")

Médiane du nombre d'avis : 3.0


La logique de cette requête est la suivante :

- On selectionne la colonne numero_of_review
- On lui applique la mediane
- On transforme le resultat en élément (avec item()) qu'on retourne dans une variable

### 3. Médiane du nombre d'avis par catégorie d'hôte

In [32]:
mediane_par_categorie = (
    df
    .with_columns([
        pl.col("number_of_reviews"),
        pl.col("host_is_superhost")
        .map_elements(lambda x: "Superhost" if x == "t" else "Hôte standard", return_dtype=pl.String)
        .alias("number_of_reviews")
    ])
    .group_by("categorie_hote")
    .agg(
        pl.col("number_of_reviews").median().alias("mediane_avis")
    )
    .sort("mediane_avis", descending=True)
)
print(mediane_par_categorie)

shape: (2, 2)
┌────────────────┬──────────────┐
│ categorie_hote ┆ mediane_avis │
│ ---            ┆ ---          │
│ str            ┆ f64          │
╞════════════════╪══════════════╡
│ Superhost      ┆ 24.0         │
│ Hôte standard  ┆ 2.0          │
└────────────────┴──────────────┘


La logique de cette requête est la suivante :
- On se sert des colonnes déjà présentes pour en créer d'autres (with column)
- Les deux colonnes crées sont number_of_reviews (incahngés) et number_of_reviews grâce aux calculs effectués sur les autres colonnes
- On groupe par la categorie d'hote (group by) nouvellement crée.
- Sur lequel on utilise des fonctions d'aggregations (.agg) seulement sur la colonne number_of_reviews (mediane), l'autre colonne sert juste d'intitulé pour afficher les résultats.
- Enfin on trie les résultats sur la médiane calculé précedemment. Ce qui nous laisse 2 colonnes avec le 'categorie_hote' qui a permi le group by.

### 4. Densité de logements par quartier de Paris

In [33]:
densite_quartier = (
    df
    .group_by("neighbourhood_cleansed")
    .agg(
        pl.len().alias("nb_logements")
    )
    .with_columns([
        (pl.col("nb_logements") / pl.col("nb_logements").sum() * 100)
        .round(2)
        .alias("proportion_pct")
    ])
    .sort("nb_logements", descending=True)
)
print(densite_quartier)

shape: (20, 3)
┌────────────────────────┬──────────────┬────────────────┐
│ neighbourhood_cleansed ┆ nb_logements ┆ proportion_pct │
│ ---                    ┆ ---          ┆ ---            │
│ str                    ┆ u32          ┆ f64            │
╞════════════════════════╪══════════════╪════════════════╡
│ Buttes-Montmartre      ┆ 10555        ┆ 11.01          │
│ Popincourt             ┆ 8430         ┆ 8.79           │
│ Vaugirard              ┆ 7802         ┆ 8.14           │
│ Batignolles-Monceau    ┆ 6857         ┆ 7.15           │
│ Entrepôt               ┆ 6558         ┆ 6.84           │
│ …                      ┆ …            ┆ …              │
│ Élysée                 ┆ 2898         ┆ 3.02           │
│ Hôtel-de-Ville         ┆ 2821         ┆ 2.94           │
│ Palais-Bourbon         ┆ 2740         ┆ 2.86           │
│ Luxembourg             ┆ 2701         ┆ 2.82           │
│ Louvre                 ┆ 2026         ┆ 2.11           │
└────────────────────────┴──────────────┴

La logique de cette requête est la suivante :
- On commence par un groupby sur le quartier sur lequel on fait immediatement une fonction d'agregation sur le nombre d'individus (donc de logements) par quartier.
- Ce qui nous laisse deux colonnes : le nom du quartier et le nombre de logements par quartier.
- On se sert de ces deux colonnes pour en créer une nouvelle (sans supprimer les autres) à savoir la proportion de logements du quartier sur le nombre de logements total (grâce à sum()).
- On nomme cette nouvelle colonne proportion_pct.
- On trie sur cette nouvelle colonne.

### 5. Quartiers avec le plus fort taux de réservation par mois

In [34]:
taux_quartier = (
    df
    .with_columns([
        ((365 - pl.col("availability_365")) / 365 * 100)
        .alias("taux_reservation_pct")
    ])
    .group_by("neighbourhood_cleansed")
    .agg([
        pl.col("taux_reservation_pct").mean().round(2).alias("taux_moyen_pct"),
        pl.len().alias("nb_logements")
    ])
    .sort("taux_moyen_pct", descending=True)
    .head(10)
)
print(taux_quartier)

shape: (10, 3)
┌────────────────────────┬────────────────┬──────────────┐
│ neighbourhood_cleansed ┆ taux_moyen_pct ┆ nb_logements │
│ ---                    ┆ ---            ┆ ---          │
│ str                    ┆ f64            ┆ u32          │
╞════════════════════════╪════════════════╪══════════════╡
│ Ménilmontant           ┆ 71.08          ┆ 5271         │
│ Buttes-Chaumont        ┆ 69.73          ┆ 5465         │
│ Buttes-Montmartre      ┆ 69.25          ┆ 10555        │
│ Entrepôt               ┆ 68.99          ┆ 6558         │
│ Popincourt             ┆ 68.87          ┆ 8430         │
│ Gobelins               ┆ 68.26          ┆ 3304         │
│ Reuilly                ┆ 67.74          ┆ 4003         │
│ Panthéon               ┆ 66.36          ┆ 3010         │
│ Vaugirard              ┆ 65.9           ┆ 7802         │
│ Observatoire           ┆ 65.16          ┆ 3611         │
└────────────────────────┴────────────────┴──────────────┘


La logique de cette requête est la suivante :
- On se sert d'une colonne déjà présente (availability_365) pour calculer le taux de réservation de chaque logement sur l'année (en proportion de jours).
- La colonne crée s'appelle taux_reservation_pct.
- On groupe par le quartier (neighbourhood_cleansed).
- Sur lequel on utilise des fonctions d'aggregations (.agg) sur la colonne taux_reservation_pct (mean - moyenne) et sur le nombre de logements (len - taille) que l'on nomme respectivement en taux_moyen_pct et nb_logements. Vu que c'est un group by, on ne garde que la colonne qui a servi à faire les groupes (neighbourhood_cleansed) ainsi que les colonnes qui sont les résultats de l'aggrégation (taux_moyen_pct et nb_logements).
- Enfin on trie les résultats sur la moyenne calculé précedemment (taux_moyen_pct). Ce qui nous laisse au final 3 colonnes avec le 'neighbourhood_cleansed' qui a permi le group by.

## Etape 3 : Connectez votre base à un outil de business intelligence

J'ai rencontré beaucoup de difficultés à procéder à cette connection.
Dans un premier temps j'ai choisi d'utiliser l'outil Tableau car je n'ai pas d'abonnement Office 365. Cependant, cela m'a posé plus de difficultés que prévu :
- 1) l'outil Tableau public (dans sa forme gratuite) n'a pas d'option pour se connecter à mongodb, ni à un serveur sql. Les 3 seules options qu'il y avait étaient les suivantes :
  - Connecteur de données Web
  - Google Drive
  - OData
Selon CLaude il n'y avait pas moyen de poursuivre avec cette version gratuite. Par conséquent j'ai installé la version Desktop avec un essai de 14 jours gratuit.
- 2) Par conséquent, j'ai installé la version payante (Tableau desktop) avec un essai gratuit de 14 jours. En effet CLaude me disait qu'il y avait une connection à un serveur mysql de prévu, et qu'avec un utilitaire (mongosqld), je pourrai faire l'intermediaire entre mon mongodb et la connection sql. Voici les étapes :
  - Télécharger le connecteur MongoDB BI connector : Il traduit les requêtes SQL de Tableau en requêtes MongoDB. Donc à priori, les requêtes de Tableau sont écrites nativement en SQL et il faut donc un "traducteur" pour les transformer en requêtes mongodb. Celui-ci est à aller chercher directement sur le site de mongodb. Il faut choisir la bonne version de Ubuntu qui correspond à celle utilisé par wsl.
  - Installer l'utilitaire (mongosqld) : Dans la mesure où c'est une archive .tgz que j'ai téléchargé (et non un .deb), il faut extraire le contenu de l'archive et se rendre dans le dossier extrait, au répértoire bin afin de lancer le service :
  - cd mongodb-bi-linux-x86_64-ubuntu2404-v2.14.28/
  - ls bin/
  - chmod +x ./bin/mongosqld : si nécessaire, pour donner les droits d'execution depuis linux (lorsque c'est depuis Windows qu'on extrait l'archive, on a pas forcement les droits d'éxecution depuis linux)
  - ./bin/mongosqld --mongo-uri "mongodb://127.0.0.1:27017" --addr 127.0.0.1:3307
- 3) Une fois que le service de traduction est correctement installé, j'ai donc essayé de me connecter à mongodb via le service de connexion MySQL depuis l'utilitaire. Or j'ai eu le message d'erreur suivant au moment de la connexion (dans le formulaire de connexion où il faut remplir les champs) :
  - [MySQL][ODBC 9.7(w) Driver][mysqld-5.7.12 mongosqld v2.14.28]This command is not supported in the prepared statement protocol yet
Impossible de se connecter au serveur MySQL “localhost:3307”. Vérifiez que le serveur est en cours d’exécution et que vous disposez des droits nécessaires pour accéder à la base de données demandée.
  - Cela est "apparamment" du au fait que j'ai initialement installé un driver odbc sur Windows pour tenter de réaliser une autre solution proposé par CLaude :
      - Créer une connexion entre l'utilitaire Windows "Sources de données ODBC" et Mongodb sous linux, qui soit fixe et qui puisse demeurer en tant qu'objet ré-utilisable par d'autres services.
      - Connecter Tableau et Mongodb en utilisant l'objet ODBC crée précedemment tel quel, sans avoir à réaliser plus de paramétrages. Au final sera aura fait le lien suivant : Tableau (via l'onglet Autre base de données (ODBC)) --> Objet DSN de connexion vers mongodb (plus particulièrement vers la base de données noscités) --> Mongodb sous linux (base de données nos cités).
      - Cela partait du principe qu'un outil installé sous Windows ne pouvait pas directement se connecter à mongodb en localhost sans passer par un intermedaire. Ce qui était faux puisque Compass le faisait directement.
    - Cela n'a pas été possible puisque le driver intialement installé correspondait à la dernière version du driver (9.*) odbc pour mongodb alors que notre utilitaire traducteur (mongosqld) était fait pour fonctionner avec la version 8.*, donc non compatible. Il fallait desinstaller la dernière version (9.*) pour installer la 8.*, ce que j'ai essayé de faire mais qui n'a pas marché car pour installer cette version, il fallait Visual Studio (et c'est là où j'ai laissé tomber).
- 4) Par chance, avant que j'abandonne définitivement, par chance, j'ai vu qu'il y avait un connecteur natif dans la version Desktop de Tableau qui permettait de se connecter directement à MongoDB. Au passage, je crois que cette connexion utilise un des drivers que j'ai installé ce jour (Atlas SQL ODBC 2.0.7, Connector for Bi v2.14,28, MongoDB ODBC 1.4.8) mais je ne sais pas lequel.
    - J'ai donc utilisé cette fenêtre pour renseigner les informations de mon serveur mongodb, accessible par l'url "mongodb://127.0.0.1:27017". Cependant ces informations ne marchait pas. Car en effet, quel que soit le connecteur utilisé, Tableau ne gère que du sql. Donc il faut se connecter non pas directement au serveur mongodb mais au service que l'on a lancé précedemment (monsqld) et gère la traduction.
    - En effet, le connecteur MongoDB BI se connecte au service MongoDB BI Connector (mongosqld - port 3307) et accède grâce à cela aux information de mongodb. Par conséquent, les étapes sont les suivantes :
          - Lancer le service mongosqld via la commande : ./bin/mongosqld --mongo-uri "mongodb://127.0.0.1:27017" --addr 127.0.0.1:3307.
          - Utiliser l'onglet "Connecteur MongoDB Bi" Dans Tableau Desktop.
          - Renseigner les informations suivantes : Serveur : localhost - Port : 3307 - Base de données : noscites - Nom d'utilisateur : laisse vide -Mot de passe : laisse vide

On aurait également pu faire un export comme ci-dessous, puis le re-intégrer dans Tableau Desktop (ou tableau public) via l'onglet "Connexion à un fichier" très facilement. Mais cela n'aurait pas répondu à la demande de se connecter directement à mongodb.

In [40]:
#df = pl.DataFrame(list(collection.find({}, {"_id": 0})), infer_schema_length=None)
df.write_csv("listings_paris_export.csv")
print(f"Export terminé : {len(df)} lignes")

Export terminé : 95885 lignes


# Exercice partie 3 - Concevez votre base de données

## Etape 1 : Importez les données dans une même collection

Ci-dessous les étapes pour impporter les données en faisant en sorte qu'à lissue de l'importation, on puisse distinguer les villes qui viennent de Paris et celles de Lyon :
- 1) On télécharge le fichier de données listing_lyon.csv

- 2) Dans mongosh, on ajoute un champ aux logements déjà présents pour spécifier qu'ils sont bien à Paris :

// Dans mongosh
use noscites
db.listings_paris.updateMany(
  {},
  { $set: { city: "Paris" } }
)

- 3) On importe les données du fichier csv déjà présent (listing_lyon.csv) via le bash :

mongoimport \
  --db noscites \
  --collection listings_paris \
  --type csv \
  --headerline \
  --file listings_Lyon.csv

- 4) On ajoute un champ "city" renseigné à "Lyon" pour tous les logements qui n'ont pas déjà ce champs :

// 2. Dans mongosh — ajouter city="Lyon" aux docs qui n'ont pas encore de city
db.listings_paris.updateMany(
  { city: { $exists: false } },
  { $set: { city: "Lyon" } }
)

- 5) Une alternative pour l'étape 3 et 5 aurait été de faire un script Python qui aurait permis d'importer les individus présent dans le fichier listing_lyon.csv en leur rajoutant au passage un attribut 'city' resneigné à "Lyon". Ce script aurait été le suivant (on le met en markdown pour éviter toute re-importation): 

with open("listings_Lyon.csv", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    docs = []
    for row in reader:
        row["city"] = "Lyon"
        docs.append(row)

collection.insert_many(docs, ordered=False)
print(f"{len(docs)} documents Lyon importés")

## Etape 2 : Répliquez vos données avec ReplicaSet

### Méthode 1 : en utilisant le serveur actuel sur le port 27017 comme PRIMARY

Cette méthode m'a été dictée par Claude qui me conseilait plutôt de transformer le serveur principal tournant sur le port 27017 comme PRIMARY. Cela implique quelques modifications notamment du fichier de conf de Mongod pour que cela fonctionne.

#### 1. Modifier le fichier de conf de mongod (serveur par défaut qui se lance sur le port 27017)

- sudo nano /etc/mongod.conf
- Ajouter au niveau de la balise commenaire #replication
      - replication:
          replSetName: "rs0"
- relancer le service mongod (serveur par défaut) : sudo systemctl restart mongod

Cela permet de faire en sorte que le serveur actuel soit configuré pour fonctionner avec un ReplicatSet (ce qui n'est pas le cas par défaut)

#### 2. Lancer les autres instances de serveur 

Les autres instances de serveur vont venir se "brancher" sur le même replicatSet que celui qui a été défini dans le fichier de conf.
Dans la mesure où ces instances ne récupère par nécessairement (à confirmer) les informations contenue dans le fichier mongod.conf, il faut leur spécifier exactement les paramètres pour fonctionner en concert avec le premier serveur défini par l'alias 'mongod', qui lui utilise par défaut le fichier de conf mongod.conf :
- --replSet : le nom du replSet, il faut qu'il soit le même pour tous les serveurs qui vont être à l'oeuvre de ce replicatSet.
- --port : le port du serveur, il faut que chaque serveur qui est lancé sur une même adresse ip, en l'occurence localhost ait un port différent (pour simuler des serveur différents).
- --dbpath : le répertoire dédié au serveur, sur lequel sera copié les données de réplication du serveur primaire. On ne l'a pas spécifié pour le serveur primaire car celui-ci a été lancé par systemd qui utilise les paramètres par défaut prévu par le fichier de conf /etc/mongod.conf.
- --bind_ip l'adresse ip du serveur --> Très important car on pourrait vouloir lancer le service sur serveur distant. Ce qui n'est pas le cas dans notre exemple.

- mongod --replSet rs0 --port 27018 --dbpath ~/mongodb/rs0-1 --bind_ip localhost
- mongod --replSet rs0 --port 27019 --dbpath ~/mongodb/rs0-2 --bind_ip localhost
- mongod --replSet rs0 --port 27020 --dbpath ~/mongodb/rs0-arb --bind_ip localhost

En fait, tous ces pramètres n'ont pas été spécifié pour le primary car, avec cette méthode, il a été lancé et géré par systemd (C'est d'ailleurs peut-être pour cela que le fichier de conf s'appelle mongod - pour être récupéré et executé par systemd automatiquement). Dans ce fichier voilà les pramètres prévus par défaut :
- replSetName (repleSet) : "rs0"
- dbPath: /var/lib/mongodb
- port: 27017
- bindIp: 127.0.0.1
Par conséquent il n'y a pas besoin de les spécifier mais simplement lancer la commande sudo systemctl start mongod et sytemd se charge d'aller chercher ces pramètre.

#### 3. Initialiser le ReplicaSet

- Se connecter au Primary avec mongosh (Est-ce qu'on aurait pu faire de même en se connectant à un autre serveur ?).
- initialiser le replicatSet avec la commande rs.initiate et les paramètres de configuration. On aurait également pu lancer seulement rs.initiate() et ajouter les serveur les uns après les autres mais la commande ci-dessous fait tout en un :

rs.initiate({
  _id: "rs0",
  members: [
    { _id: 0, host: "localhost:27017" },
    { _id: 1, host: "localhost:27018" },
    { _id: 2, host: "localhost:27019" },
    { _id: 3, host: "localhost:27020", arbiterOnly: true }
  ]
})

- Verifier la configuration avec rs.status() : On doit bien voir chaque port du serveur localhost associé à un type de serveur (primary, secondary ou arbiter).
- Exporter la base de données vers le Primary du Replicat Set (pas encore totalement compris le but de cette opération). Ci-dessous les paramètres :
      - --port : port d'importation ou d'exportation
      - --db : la base de données en question
      - --out : le chemin d'exportation de la base de données (qui sera exportée avec son nom si rien n'est spécifié).
Cela donne les requêtes d'exportatiion et d'importation ci-dessous :

- Exporter depuis l'ancienne instance :
mongodump --port 27017 --db noscites --out ~/backup_noscites

- Importer dans le Primary du ReplicaSet :
mongorestore --port 27017 --db noscites ~/backup_noscites/noscites

#### 4.  Vérifier la réplication

On execute les commandes ci-dessous sur chaque serveur (donc sur chaque port) pour vérifier que la replication a bien fonctionné.

// Sur le Secondary 1
mongosh --port 27018
rs.secondaryOk()
db = db.getSiblingDB("noscites")
db.listings_paris.countDocuments()
// Doit retourner le même nombre que sur le Primary

On aurait également pu faire un test pour stopper le serveur principal et voir si les autres ont pris le relais (et si par exemple le secondary est devenu le primary).

J'ai fait le test en stoppant le PRIMARY (via systemctl) avec la commande sudo systemctl stop mongod et effectivement, c'est l'un des secondary (celui sur le port 27019) qui a pris le relais (en devenant PRIMARY). Parcontre, lorsque j'ai relancé l'ancien PRIMARY avec la commande sudo systemctl start mongod, il n'est pas redevenu PRIMARY : C'est toujours celui sur le port 27019 qui était resté PRIMARY.

### Méthode 2 : en utilisant un autre serveur (port 27018) que celui actuel (port 27017) comme PRIMARY